In [69]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict,Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel,Field
import operator


In [70]:
load_dotenv()

True

In [71]:
class EvaluationSchema(BaseModel):
    feedback:str=Field(description="Detailed feedback for the essay")
    score:int=Field(description="Score out of 10",le=10,ge=0)

In [72]:
llm=ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite')

In [73]:
structed_model=llm.with_structured_output(EvaluationSchema)

In [74]:
class State(TypedDict):
    essay:str
    cotAnalysis:str #clarity of thought
    doaAnalysis:str #depth of analysis
    languageAnalysis:str #language analysis
    final_feedback:str
    individual_scores:Annotated[list[int],operator.add]
    avg_score:float

In [75]:
def cotAnalysis(state:State)->State:
    prompt=f"Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}"
    output=structed_model.invoke(prompt)
    return {"cotAnalysis":output.feedback,"individual_scores":[output.score]}

In [76]:
def doaAnalysis(state:State)->State:
    prompt=f"Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}"
    output=structed_model.invoke(prompt)
    return {"doaAnalysis":output.feedback,"individual_scores":[output.score]}

In [77]:
def languageAnalysis(state:State)->State:
    prompt=f"Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}"
    output=structed_model.invoke(prompt)
    return {"languageAnalysis":output.feedback,"individual_scores":[output.score]}

In [78]:
def summary(state:State)->State:
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["languageAnalysis"]} \n depth of analysis feedback - {state["doaAnalysis"]} \n clarity of thought feedback - {state["cotAnalysis"]}'
    overall_feedback =structed_model.invoke(prompt)

    # avg calculate
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'final_feedback': overall_feedback.feedback, 'avg_score': avg_score}

In [87]:
#define graph
graph=StateGraph(State)

#add nodes
graph.add_node("doaAnalysis",doaAnalysis)
graph.add_node("cotAnalysis",cotAnalysis)
graph.add_node("languageAnalysis",languageAnalysis)
graph.add_node("summary",summary)

#add edges
graph.add_edge(START,"doaAnalysis")
graph.add_edge(START,"cotAnalysis")
graph.add_edge(START,"languageAnalysis")
graph.add_edge("cotAnalysis","summary")
graph.add_edge("doaAnalysis","summary")
graph.add_edge("languageAnalysis","summary")
graph.add_edge("summary",END)

#compile
workflow=graph.compile()

In [84]:
essay2 = """Technology has become an important part of modern education. From online classes to digital libraries, students today have access to more learning resources than ever before. Technology can make education more flexible, interactive, and accessible, but it also creates challenges that schools and students need to address.

One major advantage of technology is easy access to information. Students can use the internet to learn about almost any topic within a few minutes. Educational videos, online courses, and digital books can also help students understand difficult concepts in different ways. This is especially useful for students who may not learn effectively through traditional classroom teaching alone.

Technology can also make learning more interactive. Teachers can use presentations, simulations, quizzes, and educational applications to make lessons more engaging. For example, a science student can use a simulation to understand how planets move or how chemical reactions occur. These experiences can sometimes make complex topics easier to understand than textbooks alone.

However, technology also has disadvantages. Students can become distracted by social media, games, and entertainment while studying. Excessive dependence on technology may also reduce face-to-face communication and critical thinking if students simply search for answers instead of trying to solve problems themselves. Furthermore, not every student has equal access to computers and reliable internet, which can create an educational gap.

Therefore, technology should not completely replace traditional education. Instead, it should be used as a tool to support teachers and students. Schools should teach students how to use technology responsibly and critically. If used properly, technology can make education more effective while still preserving the importance of human interaction and independent thinking."""

In [85]:
initial_state={"essay":essay2}
final_state=workflow.invoke(initial_state)
print(final_state['avg_score'])
print(final_state['final_feedback'])

8.666666666666666
The essay is exceptionally well-written with high grammatical accuracy, varied sentence structures, and a clear, logical organization that makes the author's arguments very easy to follow. It provides a solid and balanced analysis of the pros and cons of technology in education, supported by concrete examples such as educational simulations, accessibility, and distractions. While the clarity of thought and language quality are of a very high standard, the analysis could be further enhanced by exploring deeper socio-economic impacts and the psychological effects of screen time.
